# BrainMRI-SpatialVLM: End-to-End Walkthrough Demonstration

> **DISCLAIMER: AI-generated research draft — not a medical diagnosis.**  
> This demonstration notebook is designed strictly for research, computational neuroscience, and machine learning experimentation. It does not provide clinical diagnostic evaluations, treatment advice, or medical assessments, and must never be utilized in clinical decision-making or patient workflows.

---

## Notebook Overview
This interactive walkthrough demonstrates each modular component of the **BrainMRI-SpatialVLM** research platform:
1. **Volume Ingestion & Preprocessing**: Loading native 3D/4D NIfTI brain MRI scans.
2. **MONAI BraTS 3D Segmentation**: Predicting 3D tumor subregion masks.
3. **Quantitative Spatial Feature Extraction**: Computing descriptive non-diagnostic physical metrics.
4. **BrainGemma3D Vision-Language Draft Reporting**: Generating structured draft reports.
5. **Multi-Planar & Montage Visualizations**: Generating orthogonal slice views and occupancy plots.
6. **Full Pipeline Execution**: Orchestrating the end-to-end automated pipeline.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config
from src.mri_io import load_nifti_volume, get_volume_metadata
from src.segmentation import BraTSSegmenter
from src.spatial_features import extract_spatial_features, compare_with_reference_annotation
from src.reporting import BrainGemma3DReporter, RESEARCH_DISCLAIMER
from src.visualization import find_informative_slices, plot_orthogonal_slices, plot_slice_montage, plot_mask_occupancy
from src.pipeline import run_full_pipeline

cfg = load_config()
print("System Name:", cfg.system.name)
print("Disclaimer :", cfg.system.disclaimer)

## 1. Load Brain MRI Volume
We select either a preloaded research case from `data/raw/Task01_BrainTumour/imagesTr/` or synthesize a test scan.

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw" / "Task01_BrainTumour" / "imagesTr"
sample_files = list(raw_dir.glob("*.nii.gz"))

if sample_files:
    sample_path = sample_files[0]
    print(f"Loading sample research case: {sample_path.name}")
    nii = nib.load(str(sample_path))
    volume_data = np.asanyarray(nii.dataobj)
    affine = nii.affine
    print(f"Volume Shape: {volume_data.shape} (ndim={volume_data.ndim})")
    # Extract FLAIR sequence if 4D
    if volume_data.ndim == 4:
        mri_3d = volume_data[:, :, :, 3].astype(np.float32)
    else:
        mri_3d = volume_data.astype(np.float32)
else:
    print("Creating synthetic 3D test volume...")
    mri_3d = np.ones((64, 64, 32), dtype=np.float32) * 50.0
    affine = np.eye(4)

print("Prepared 3D Volume Shape:", mri_3d.shape)

## 2. MONAI BraTS 3D Segmentation
Execute 3D tumor segmentation or inspect predicted masks.

In [ ]:
segmenter = BraTSSegmenter(device="cpu")

# Check if model weights are present locally
if segmenter.weights_path and segmenter.weights_path.is_file():
    print(f"Loaded MONAI model bundle weights from: {segmenter.weights_path}")
else:
    print("Pretrained bundle weights not yet downloaded. Run: python scripts/download_monai_brats.py")

# For demonstration, check if reference ground-truth mask is available
labels_dir = PROJECT_ROOT / "data" / "annotations" / "Task01_BrainTumour" / "labelsTr"
ref_mask_path = labels_dir / sample_path.name if sample_files else None

if ref_mask_path and ref_mask_path.is_file():
    mask_3d = np.asanyarray(nib.load(str(ref_mask_path)).dataobj).astype(np.int32)
    print(f"Loaded reference ground-truth mask: {mask_3d.shape}")
    print(f"Unique labels present: {np.unique(mask_3d)}")
else:
    # Synthetic mask with necrotic, edema, enhancing regions
    mask_3d = np.zeros(mri_3d.shape, dtype=np.int32)
    mask_3d[20:30, 20:30, 10:16] = 2  # Edema
    mask_3d[23:27, 23:27, 12:15] = 1  # Core
    mask_3d[24:26, 24:26, 13:14] = 3  # Enhancing

## 3. Quantitative Spatial Feature Extraction
Compute physical volume in $\text{mm}^3$, bounding box, centroid, and connected components.

In [ ]:
metrics = extract_spatial_features(
    mri_volume=mri_3d,
    segmentation_mask=mask_3d,
    voxel_spacing_mm=(1.0, 1.0, 1.0),
    affine=affine,
)

print("Measurement Type:", metrics.measurement_type)
print(f"Total Mask Voxels: {metrics.total_mask_voxels:,}")
print(f"Mask Volume      : {metrics.mask_volume_mm3:,.1f} mm^3")
print(f"Volume Fraction  : {metrics.volume_fraction_percent:.3f}%")
print(f"Centroid (Voxel) : {metrics.centroid_voxel}")
print(f"Connected Clusters: {metrics.connected_components['count']}")

## 4. BrainGemma3D Vision-Language Draft Report
Generate structured draft findings and impression using BrainGemma3D adapter.

In [ ]:
reporter = BrainGemma3DReporter(require_gpu=False)
is_avail, missing = reporter.check_components_availability()

if is_avail:
    print("BrainGemma3D local components available. Generating draft report...")
    report = reporter.generate_report(mri_3d, mask=mask_3d, spatial_features=metrics)
    print(report.to_markdown())
else:
    print("BrainGemma3D Model Status:")
    print("  ", MODEL_UNAVAILABLE_MESSAGE)
    print("Missing requirements:")
    for m in missing:
        print(f"   - {m}")

## 5. Research Visualizations
Generate orthogonal slices, multi-slice montage, and slice-wise occupancy curves.

In [ ]:
out_viz_dir = PROJECT_ROOT / "outputs" / "visualizations"
out_viz_dir.mkdir(parents=True, exist_ok=True)

# Tri-planar Orthogonal View
fig_ortho, ortho_path = plot_orthogonal_slices(
    mri_volume=mri_3d,
    mask=mask_3d,
    output_path=out_viz_dir / "walkthrough_orthogonal.png",
)
print(f"Saved Orthogonal View: {ortho_path.name}")

# Multi-slice Montage
fig_montage, montage_path = plot_slice_montage(
    mri_volume=mri_3d,
    mask=mask_3d,
    plane="axial",
    num_slices=12,
    output_path=out_viz_dir / "walkthrough_montage.png",
)
print(f"Saved Axial Montage: {montage_path.name}")

# Slice-wise Occupancy Profile
fig_occ, occ_path = plot_mask_occupancy(
    mask=mask_3d,
    output_path=out_viz_dir / "walkthrough_occupancy.png",
)
print(f"Saved Occupancy Profile: {occ_path.name}")

## 6. End-to-End Multimodal Pipeline Execution
Run `run_full_pipeline` to execute all steps and package unified JSON results.

In [ ]:
pipeline_res = run_full_pipeline(
    input_nifti_path=sample_path if sample_files else sample_path,
    config=cfg,
)

print("Pipeline Status :", pipeline_res.status.upper())
print("Case ID         :", pipeline_res.case_id)
print("Generated Paths :", json.dumps({k: str(v) for k, v in pipeline_res.paths.items() if v}, indent=2))
print("Disclaimer      :", pipeline_res.disclaimer)